In [1]:
### Import Libraries
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

# Workout - Parking Tickets PDX (Pre Cleaned)

![Cars Parked in the City of Portland](../images/stock/pexels-brett-sayles-3995715.jpg)
image credit: [Pexels](https://www.pexels.com)

Ask and you shall receive...

Early this term, I requested a years worth of parking data from [Portland's Public Records](https://www.portland.gov/public-records). With in a week, I was sent 250,000 rows of parking ticket data!

# 0. Importing and Inspecting the Data

The files used for this workout is located at the following path:

* Parking Violations CSV - `../data/parking_tickets_pdx/C455259_Data.csv`
* Parking Violation Codes XLSX - `../data/parking_tickets_pdx/Offences.xlsx`

Import the data and do the usual: 

- [ ] Inspect the dataset, looking for issues and quirks
- [ ] Preview a few rows of data

In [2]:
### Your Solution
tickets_file = "../data/parking_tickets_pdx/C455259_Data.csv"
offenses_file = "../data/parking_tickets_pdx/Offences.xlsx"

tickets_df = pd.read_csv(tickets_file)
offenses_df = pd.read_excel(offenses_file)

/Users/mmw23/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


## 0.1 Parking Tickets Overview

Inspect the Parking Tickets DataFrame

In [3]:
tickets_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Ticket no.        249944 non-null  object 
 1   District          250000 non-null  object 
 2   Street            250000 non-null  object 
 3   House no          26894 non-null   object 
 4   Local detail      6040 non-null    object 
 5   between           217123 non-null  object 
 6   and               217123 non-null  object 
 7   GPS               249472 non-null  object 
 8   From (Date-Time)  250000 non-null  object 
 9   To (Date-Time)    250000 non-null  object 
 10  State             233628 non-null  object 
 11  Vehicle type      0 non-null       float64
 12  Vehicle make      249999 non-null  object 
 13  Offense 1         250000 non-null  int64  
 14  Amount            250000 non-null  int64  
dtypes: float64(1), int64(2), object(12)
memory usage: 28.6+ MB


## 0.2 Offenses Overview

Inspect the Offenses DataFrame.

In [4]:
offenses_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Offence code  95 non-null     int64 
 1   Amount        95 non-null     int64 
 2   Long text     95 non-null     object
dtypes: int64(2), object(1)
memory usage: 2.4+ KB


# 1. Cleaning

Investigate the dataset and perform all of the necessary cleaning and modifications.

Make sure that you have at least handled the following:

- [ ] Null Values
- [ ] Data Type Conversions
- [ ] Column Naming Consistency

__Consideration:__

* How will you handle the `GPS` column?
* How might you make  use of the Violation Codes in the `Offenses.xlsx` file and the `Offense 1` column in the larger dataset?

## 1.1 Standardizing Columns

Standardize the columns of both DataFrames. 

Consider the following:
* removing leading and trailing whitespace
* converting to all lower case
* replacing blankspaces with underscores

In [5]:
# Standardize Columns
def standardize_columns(dataframe):
    dataframe.columns = (
        dataframe.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("offence", "offense")
        .str.replace(r"\.$", "", regex=True)
    )

    return dataframe
    
tickets_df = standardize_columns(tickets_df)
offenses_df = standardize_columns(offenses_df)

In [6]:
# Tickets Dataframe
tickets_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   ticket_no         249944 non-null  object 
 1   district          250000 non-null  object 
 2   street            250000 non-null  object 
 3   house_no          26894 non-null   object 
 4   local_detail      6040 non-null    object 
 5   between           217123 non-null  object 
 6   and               217123 non-null  object 
 7   gps               249472 non-null  object 
 8   from_(date-time)  250000 non-null  object 
 9   to_(date-time)    250000 non-null  object 
 10  state             233628 non-null  object 
 11  vehicle_type      0 non-null       float64
 12  vehicle_make      249999 non-null  object 
 13  offense_1         250000 non-null  int64  
 14  amount            250000 non-null  int64  
dtypes: float64(1), int64(2), object(12)
memory usage: 28.6+ MB


In [7]:
# Offenses Dataframe
offenses_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   offense_code  95 non-null     int64 
 1   amount        95 non-null     int64 
 2   long_text     95 non-null     object
dtypes: int64(2), object(1)
memory usage: 2.4+ KB


## 1.2 Merging Dataframes

Merge both dataframes.

Consider:
* joining on the `offense_1` column of the parking tickets dataframe and   `offense_code` of the offenses dataframe
* dropping columns you may no longer need after the join is complete

Note:
* the `offense_code` column may have a different name if it wasn't changed during the column cleaning step.

In [8]:
# Merge the two dataframe
parking_df_clean = pd.merge(
    tickets_df,
    offenses_df[["offense_code","long_text"]],
    left_on="offense_1",
    right_on="offense_code",
    how="left"
).drop(columns=["offense_1", "offense_code"])

parking_df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   ticket_no         249944 non-null  object 
 1   district          250000 non-null  object 
 2   street            250000 non-null  object 
 3   house_no          26894 non-null   object 
 4   local_detail      6040 non-null    object 
 5   between           217123 non-null  object 
 6   and               217123 non-null  object 
 7   gps               249472 non-null  object 
 8   from_(date-time)  250000 non-null  object 
 9   to_(date-time)    250000 non-null  object 
 10  state             233628 non-null  object 
 11  vehicle_type      0 non-null       float64
 12  vehicle_make      249999 non-null  object 
 13  amount            250000 non-null  int64  
 14  long_text         249944 non-null  object 
dtypes: float64(1), int64(1), object(13)
memory usage: 28.6+ MB


## 1.3 Dropping Columns

Are there columns that should be completely dropped? Take some time to either remove some columns or skip this step and deal with the columns later in some other way.

In [9]:
### Your Solution

# Drop Columns
columns_to_drop = ["local_detail", "vehicle_type"]

parking_df_clean = parking_df_clean.drop(
    columns= columns_to_drop
)

parking_df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   ticket_no         249944 non-null  object
 1   district          250000 non-null  object
 2   street            250000 non-null  object
 3   house_no          26894 non-null   object
 4   between           217123 non-null  object
 5   and               217123 non-null  object
 6   gps               249472 non-null  object
 7   from_(date-time)  250000 non-null  object
 8   to_(date-time)    250000 non-null  object
 9   state             233628 non-null  object
 10  vehicle_make      249999 non-null  object
 11  amount            250000 non-null  int64 
 12  long_text         249944 non-null  object
dtypes: int64(1), object(12)
memory usage: 24.8+ MB


## 1.4 Fill NaN (Categorical)

Replace the null values of the categorical columns.

***Consider creating a null map*** for a clean solution.

In [10]:
# Columns to Fill
columns_to_fill = [
    "ticket_no",
    "house_no",
    "between",
    "and",
    "gps",
    "long_text",
    "state",
    "vehicle_make"
]

# Dictionary for Filling NaNs
fill_map = {column: "UNKNOWN" for column in columns_to_fill}

parking_df_clean = parking_df_clean.fillna(fill_map)

parking_df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   ticket_no         250000 non-null  object
 1   district          250000 non-null  object
 2   street            250000 non-null  object
 3   house_no          250000 non-null  object
 4   between           250000 non-null  object
 5   and               250000 non-null  object
 6   gps               250000 non-null  object
 7   from_(date-time)  250000 non-null  object
 8   to_(date-time)    250000 non-null  object
 9   state             250000 non-null  object
 10  vehicle_make      250000 non-null  object
 11  amount            250000 non-null  int64 
 12  long_text         250000 non-null  object
dtypes: int64(1), object(12)
memory usage: 24.8+ MB


## 1.5 Latitude and Longitude

Clean up the latitude and longitude columns.

1. Create separate columns for each coordinate
2. Split the `gps` column into the new columns created in the previous step
3. Convert  the new coordinate columns into numeric datatypes
4. Drop the `gps` column now that you have a separate column for longitude and latitude.

In [11]:
# Create Latitude and Longitude Columns
parking_df_clean[["latitude", "longitude"]] = parking_df_clean["gps"].str.split(
    ";",
    expand=True
)

# Apply Numeric Conversion to Both Columns
parking_df_clean[["latitude","longitude"]] = parking_df_clean[["latitude","longitude"]].apply(
    pd.to_numeric, 
    errors="coerce"
)

# Drop the GPS Column
parking_df_clean = parking_df_clean.drop(
    columns="gps"
)

parking_df_clean.sample(10)

,ticket_no,district,street,house_no,between,and,from_(date-time),to_(date-time),state,vehicle_make,amount,long_text,latitude,longitude
199990,HA37303572,SW,4TH AVE,UNKNOWN,MILL ST,MONTGOMERY ST,8/10/2024 23:18,8/10/2024 23:19,OR,JEEP,155,FIRE HYDRANT,45.513517,-122.676199
223911,HA37349665,N,GANTENBEIN AVE,UNKNOWN,SKIDMORE ST,SHAVER ST,10/2/2024 10:58,10/2/2024 10:59,WA,DODGE,145,FAILURE TO DISPLAY CURRENT REGISTRATION (91 OR...,45.551150,-122.669061
205604,HA37329510,SW,12TH AVE,UNKNOWN,MARKET ST,MONTGOMERY ST,8/21/2024 15:52,8/21/2024 15:54,AZ,HYUNDAI,65,NO METER RECEIPT,45.514388,-122.687509
58577,HA37164049,NE,SANDY BLVD,UNKNOWN,UNKNOWN,UNKNOWN,8/11/2023 13:24,8/11/2023 13:24,OR,SUBARU,95,LOADING ZONE,45.537980,-122.614961
144182,HA37249185,SE,SALMON ST,UNKNOWN,44TH AVE,45TH AVE,4/4/2024 18:00,4/4/2024 18:02,OR,UNKNOWN,85,BLOCKING VIEW AT INTERSECTION,45.504753,-122.625592
215909,HA37318819,NW,COUCH ST,UNKNOWN,12TH AVE,13TH AVE,9/16/2024 18:21,9/16/2024 18:22,OR,KIA,65,NO METER RECEIPT,45.524004,-122.683798
121639,HA37240037,NW,9TH AVE,UNKNOWN,BURNSIDE ST,COUCH ST,2/7/2024 10:05,2/7/2024 10:05,WA,KIA,85,YELLOW CURB,45.523254,-122.680215
227174,HA37337017,S,ABERNETHY ST,UNKNOWN,RIVER PKY,BOND AVE,10/8/2024 10:37,10/8/2024 10:37,OR,SUBARU,65,NO METER RECEIPT,45.495055,-122.670207
207850,HA37321327,NW,GLISAN ST,UNKNOWN,11TH AVE,12TH AVE,8/27/2024 11:10,8/27/2024 11:12,OR,VOLVO,65,NO METER RECEIPT,45.526023,-122.682241
67409,HA37174062,SW,9TH AVE,UNKNOWN,UNKNOWN,UNKNOWN,9/6/2023 10:14,9/6/2023 10:15,WA,MERCEDES BENZ,145,FAILURE TO DISPLAY CURRENT REGISTRATION (91 OR...,45.496826,-122.690084


## 1.6 Missing Latitude and Longitude

Fill the any missing latitude and longitude values.

Consider the following:
* Would replacing the missing coordinates with the median of an entire column be the best course of action?
* How might we get a more accurate (but not perfect) median value to replace missing values with?

In [12]:
# Set Up Columns
columns_to_fix = ["latitude", "longitude"]

# Apply Median Value based on Distict Groups
for column in columns_to_fix:
    parking_df_clean[column] = parking_df_clean[column].fillna(
        parking_df_clean.groupby("district")[column].transform("median")
    )

parking_df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   ticket_no         250000 non-null  object 
 1   district          250000 non-null  object 
 2   street            250000 non-null  object 
 3   house_no          250000 non-null  object 
 4   between           250000 non-null  object 
 5   and               250000 non-null  object 
 6   from_(date-time)  250000 non-null  object 
 7   to_(date-time)    250000 non-null  object 
 8   state             250000 non-null  object 
 9   vehicle_make      250000 non-null  object 
 10  amount            250000 non-null  int64  
 11  long_text         250000 non-null  object 
 12  latitude          250000 non-null  float64
 13  longitude         250000 non-null  float64
dtypes: float64(2), int64(1), object(11)
memory usage: 26.7+ MB


## 1.7 DateTime Conversions

Convert the time-related columns to date time objects

In [13]:
# Datetime Conversion
columns_to_fix = ["from_(date-time)", "to_(date-time)"]

parking_df_clean[columns_to_fix] = parking_df_clean[columns_to_fix].apply(
    pd.to_datetime,
    errors="coerce",
    format="%m/%d/%Y %H:%M"
)

parking_df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   ticket_no         250000 non-null  object        
 1   district          250000 non-null  object        
 2   street            250000 non-null  object        
 3   house_no          250000 non-null  object        
 4   between           250000 non-null  object        
 5   and               250000 non-null  object        
 6   from_(date-time)  250000 non-null  datetime64[ns]
 7   to_(date-time)    250000 non-null  datetime64[ns]
 8   state             250000 non-null  object        
 9   vehicle_make      250000 non-null  object        
 10  amount            250000 non-null  int64         
 11  long_text         250000 non-null  object        
 12  latitude          250000 non-null  float64       
 13  longitude         250000 non-null  float64       
dtypes: d

## 1.8 Cleaning the Streets

The street columns, specially the `between` and `and` columns are a bit strange. How might you transform the data to make it make sense in terms of the column names?

In [14]:
# Create Cross Streets Column
parking_df_clean["intersection"] = "BETWEEN " + parking_df_clean["between"] + " AND " + parking_df_clean["and"]

# Drop Between and And
columns_to_drop = ["between", "and"]

parking_df_clean = parking_df_clean.drop(
    columns=columns_to_drop
)

parking_df_clean["intersection"].sample(10)

2763          BETWEEN MAIN ST AND JEFFERSON ST
249036            BETWEEN 6TH AVE AND BROADWAY
249107             BETWEEN UNKNOWN AND UNKNOWN
102017             BETWEEN UNKNOWN AND UNKNOWN
161009       BETWEEN OAK ST AND HARVEY MILK ST
78470     BETWEEN JEFFERSON ST AND COLUMBIA ST
17417       BETWEEN EVERETT ST AND FLANDERS ST
85089            BETWEEN 13TH AVE AND 14TH AVE
228548           BETWEEN HOYT ST AND IRVING ST
64614      BETWEEN MONTGOMERY ST AND RIVER PKY
Name: intersection, dtype: object

## 1.9 Rename Columns (optional)

Rename any other columns for spelling or clarity.

In [15]:
# Rename Columns
rename_map = {
    "from_(date-time)": "start_time",
    "to_(date-time)": "end_time",
    "long_text": "violation",
    "house_no": "house_number",
    "ticket_no": "ticket_id",
    "state": "license_plate_state",
    "amount": "fine_amount"
}

parking_df_clean = parking_df_clean.rename(columns=rename_map)

parking_df_clean.columns

Index(['ticket_id', 'district', 'street', 'house_number', 'start_time',
       'end_time', 'license_plate_state', 'vehicle_make', 'fine_amount',
       'violation', 'latitude', 'longitude', 'intersection'],
      dtype='object')

## 1.10 Column Order (optional)

Rearrange the column order as you would like.

In [ ]:
# Preferred Order
column_order = [
    "ticket_id",
    "start_time",
    "end_time",
    "district",
    "street",
    "intersection",
    "latitude",
    "longitude",
    "vehicle_make",
    "license_plate_state",
    "violation",
    "fine_amount",
]

parking_df_clean = parking_df_clean[column_order]

parking_df_clean.info()

# 2 Vehicles

## 2.1 __Common Cars__

Which `vehicle make` appears the most? Why might this be?

In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

## 2.2 __Exotic Cars__

How many Lamborghinis and Ferraris appear in this dataset?

Are there any other exotic cars in the data?


In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

# 3. Fees and Violations

## 3.1 What are the most common violations?

In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

## 3.2 What is the total for all fees owed for based on the available data?

In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

## 3.3 Which violations have the largest total fees for the entire dataset?

In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

# 4. Location Data

## 4.1 Which Districts Are Ticketed the Most?

How does each district rank in terms of Parking Ticket Count?

In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

## 4.2 Ticket Hotspots

Visualize the parking ticket hotspots.
* Which areas are the hottest?
* Which areas are the coldest?
* Why might this be?

In [ ]:
## Your Solution




















### Your Observation(s)

*
*
*

## 4.3 Most Ticketed Street

Which streets have the most tickets?

In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

## Most Common Violations on the Most Ticketed Street

Which violations are the most common the this street?

In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

# 5. Time

## 5.1 Are there any patterns to the hours in tickets are issued?

Consider creating a new column for this task:
* hour (`dt.hour`)

In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

## 5.2 Are there any patterns to the days in tickets are issued?

Consider creating a new column for this task:
* day (`dt.day_name()`)

In [ ]:
## Your Solution










### Your Observation(s)

*
*
*

# 6. More to Explore (Optional)

There are still more insights to discover. Add cells below to make new discoveries or to answer your unique questions based on this dataset.

# 7. Reflection

Take some time to think, reflect, and discuss on this activity.

* What did you learn today that was new for you?
* Will the data and insights gained from today change how you go about your day-to-day in any way?
    * If so, how?
    * If not, why not?
* How might today's discoveries be useful to people?
* Did you label the axes of your visualizations?